# ਪਾਠ 10 - ਉਤਪਾਦਨ ਵਿੱਚ ਏਆਈ ਏਜੰਟ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ Microsoft Agent Framework (Python) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਏਆਈ ਏਜੰਟਾਂ ਲਈ **ਉਤਪਾਦਨ ਪੈਟਰਨ** ਸਿਖੋਗੇ। ਅਸੀਂ ਇਹ ਕਵਰ ਕਰਦੇ ਹਾਂ:

- **ਨਿਰੀਖਣਯੋਗਤਾ** — ਏਜੰਟ ਅੰਤਰਕਿਰਿਆਵਾਂ ਵਿੱਚ ਟਾਈਮਿੰਗ ਅਤੇ ਲਾਗਿੰਗ ਸ਼ਾਮਲ ਕਰਨਾ
- **ਮੁਲਾਂਕਣ** — ਪ੍ਰਤਿਕ੍ਰਿਆ ਦੀ ਗੁਣਵੱਤਾ ਨੂੰ ਸਕੋਰ ਕਰਨ ਲਈ ਇੱਕ ਮੁਲਾਂਕਣ ਏਜੰਟ ਦੀ ਵਰਤੋਂ
- **ਲਾਗਤ ਪ੍ਰਬੰਧਨ** — ਟੋਕਨ ਅਪਟੀਮਾਈਜ਼ੇਸ਼ਨ ਅਤੇ ਮਾਡਲ ਚੋਣ ਲਈ ਰਣਨੀਤੀਆਂ

ਦ੍ਰਿਸ਼ਟਾਂਤ ਇੱਕ **ਟ੍ਰੈਵਲ ਏਜੰਟ** ਹੈ ਜੋ ਉਪਭੋਗਤਾਵਾਂ ਨੂੰ ਯਾਤਰਾ ਦੀ ਯੋਜਨਾ ਬਣਾਉਣ ਵਿੱਚ ਮਦਦ ਕਰਦਾ ਹੈ, ਜਿਸ ਤੇ ਨਿਗਰਾਨੀ ਅਤੇ ਮੁਲਾਂਕਣ ਲਗਾਇਆ ਗਿਆ ਹੈ।


## ਸੈਟਅਪ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ਉਤਪਾਦਨ ਬਾਰੇ ਵਿਚਾਰ

AI ਏਜੰਟਾਂ ਨੂੰ ਪ੍ਰੋਟੋਟਾਇਪ ਤੋਂ ਉਤਪਾਦਨ ਵਿੱਚ ਲਿਜਾਣ ਲਈ ਤਿੰਨ ਸਤੰਭਾਂ 'ਤੇ ਧਿਆਨ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ:

1. **ਦੇਖਣਯੋਗਤਾ** — ਤੁਹਾਨੂੰ ਇਹ ਵੇਖਣ ਦੀ ਸਮਰੱਥਾ ਚਾਹੀਦੀ ਹੈ ਕਿ ਏਜੰਟ ਕੀ ਕਰ ਰਿਹਾ ਹੈ, ਇਸਨੂੰ ਕਿੰਨਾ ਸਮਾਂ ਲੱਗਦਾ ਹੈ, ਅਤੇ ਇਹ ਕਿਹੜੇ ਟੂਲਾਂ ਨੂੰ ਕਾਲ ਕਰਦਾ ਹੈ। ਟ੍ਰੇਸਿੰਗ ਅਤੇ ਲੌਗਿੰਗ ਦੇ ਬਿਨਾਂ, ਉਤਪਾਦਨ ਸਮੱਸਿਆਵਾਂ ਦੀ ਡੀਬੱਗਿੰਗ ਲਗਭਗ ਅਸੰਭਵ ਹੈ।

2. **ਮੁਲਾਂਕਣ** — ਆਟੋਮੇਟਿਡ ਗੁਣਵੱਤਾ ਜਾਂਚ ਇਹ ਯਕੀਨੀ ਬਣਾਉਂਦੀਆਂ ਹਨ ਕਿ ਏਜੰਟ ਦੇ ਜਵਾਬ ਸਮੇਂ ਦੇ ਨਾਲ ਸਹੀ, ਪੂਰੇ ਅਤੇ ਮਦਦਗਾਰ ਰਹਿੰਦੇ ਹਨ। ਇੱਕ ਮੁਲਾਂਕਣ ਏਜੰਟ ਨਿਰਧਾਰਤ ਮਿਆਰਾਂ ਦੇ ਖ਼ਿਲਾਫ਼ ਜਵਾਬਾਂ ਨੂੰ ਅੰਕ ਦੇ ਸਕਦਾ ਹੈ।

3. **ਲਾਗਤ ਪ੍ਰਬੰਧਨ** — ਟੋਕਨਾਂ ਦੀ ਵਰਤੋਂ ਸਿੱਧਾ ਲਾਗਤ ਨੂੰ ਪ੍ਰਭਾਵਿਤ ਕਰਦੀ ਹੈ। ਪ੍ਰਾਂਪਟ ਅਪਟੀਮਾਈਜ਼ੇਸ਼ਨ, ਮਾਡਲ ਚੋਣ ਅਤੇ ਕੈਸ਼ਿੰਗ ਵਰਗੀਆਂ ਰਣਨੀਤੀਆਂ ਖਰਚਿਆਂ ਨੂੰ ਨਿਯੰਤਰਣ ਹੇਠ ਰੱਖਣ ਵਿੱਚ ਮਦਦ ਕਰਦੀਆਂ ਹਨ ਬਿਨਾਂ ਗੁਣਵੱਤਾ ਨੂੰ ਕੁਰਬਾਨ ਕੀਤੇ।


## ਇੱਕ ਦੇਖਣਯੋਗ ਏਜੰਟ ਬਣਾਉਣਾ

ਅਸੀਂ ਯਾਤਰਾ ਦੇ ਔਜ਼ਾਰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਾਂ ਅਤੇ ਏਜੰਟ ਕਾਲ ਨੂੰ ਟਾਈਮਿੰਗ ਨਾਲ ਲਪੇਟਦੇ ਹਾਂ ਤਾਂ ਜੋ ਅਸੀਂ ਵਿਲੰਬ ਦੀ ਨਿਗਰਾਨੀ ਕਰ ਸਕੀਏ। ਉਤਪਾਦਨ ਵਿੱਚ ਤੁਸੀਂ OpenTelemetry ਜਾਂ ਕਿਸੇ ਸਮਾਨ ਟਰੇਸਿੰਗ ਬੈਕਐਂਡ ਨਾਲ ਇਕੀਕਰਨ ਕਰੋਗੇ।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## ਮੁਲਾਂਕਣ ਨਮੂਨੇ

ਇੱਕ ਆਮ ਪ੍ਰੋਡਕਸ਼ਨ ਪੈਟਰਨ ਇੱਕ ਦੂਜੇ ਏਜੰਟ ਨੂੰ **ਮੁਲਾਂਕਣਕਰਤਾ** ਵਜੋਂ ਵਰਤਣਾ ਹੈ। ਮੁਲਾਂਕਣਕਰਤਾ ਮੁੁੱਖ ਏਜੰਟ ਦੀ ਪ੍ਰਤੀਕ੍ਰਿਆ ਨੂੰ ਪਹਿਲਾਂ ਨਿਰਧਾਰਤ ਮਾਪਦੰਡਾਂ ਦੇ ਖ਼ਿਲਾਫ਼ ਅੰਕਤ ਕਰਦਾ ਹੈ, ਜਿਵੇਂ ਕਿ ਪੂਰਨਤਾ, ਸਹੀਤਾ, ਅਤੇ ਸਹਾਇਕਤਾ।

This enables:
- ਉਪਭੋਗਤਿਆਂ ਤੱਕ ਜਵਾਬ ਪਹੁੰਚਣ ਤੋਂ ਪਹਿਲਾਂ ਸੁਚਾਲਿਤ ਗੁਣਵੱਤਾ ਜਾਂਚ
- ਜਦੋਂ ਪ੍ਰੰਪਟ ਜਾਂ ਮਾਡਲ ਬਦਲਦੇ ਹਨ ਤਾਂ ਰਿਗ੍ਰੈਸ਼ਨ ਦੀ ਪਹਿਚਾਣ
- ਸਮੇਂ ਦੇ ਨਾਲ-ਨਾਲ ਏਜੰਟ ਦੇ ਪ੍ਰਦਰਸ਼ਨ ਦੀ ਲਗਾਤਾਰ ਨਿਗਰਾਨੀ


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ਲਾਗਤ ਪ੍ਰਬੰਧਨ ਰਣਨੀਤੀਆਂ

ਉਤਪਾਦਨ AI ਏਜੰਟਾਂ ਲਈ ਲਾਗਤਾਂ ਨੂੰ ਨਿਯੰਤਰਿਤ ਕਰਨਾ ਬਹੁਤ ਜਰੂਰੀ ਹੈ। ਇੱਥੇ ਕੁਝ ਮੁੱਖ ਰਣਨੀਤੀਆਂ ਹਨ:

| ਰਣਨੀਤੀ | ਵੇਰਵਾ |
|---|---|
| **ਪ੍ਰਾਂਪਟ ਬਿਹਤਰਕਰਨ** | ਸਿਸਟਮ ਨਿਰਦੇਸ਼ ਸੰਖੇਪ ਰੱਖੋ। ਇਨਪੁੱਟ ਟੋਕਨ ਘਟਾਉਣ ਲਈ ਅਣਲੋੜਾ ਸੰਦਰਭ ਹਟਾਓ। |
| **ਮਾਡਲ ਚੋਣ** | ਸਰਲ ਕਾਰਜਾਂ, ਜਿਵੇਂ ਵਰਗੀਕਰਨ ਜਾਂ ਤੱਥ ਨਿਕਾਸ ਲਈ ਛੋਟੇ, ਸਸਤੇ ਮਾਡਲ (e.g. GPT-4o-mini) ਵਰਤੋ, ਅਤੇ ਜਟਿਲ ਤਰਕ ਲਈ ਵੱਡੇ ਮਾਡਲ ਰੱਖੋ। |
| **ਕੈਸ਼ਿੰਗ** | ਉਪਕਰਣ ਨਤੀਜੇ ਅਤੇ ਅਕਸਰ ਆਉਣ ਵਾਲੀਆਂ ਕਵੈਰੀਆਂ ਕੈਸ਼ ਕਰੋ ਤਾਂ ਕਿ ਦੁਹਰਾਏ ਗਏ API ਕਾਲਾਂ ਤੋਂ ਬਚਿਆ ਜਾ ਸਕੇ। |
| **ਟੋਕਨ ਬਜਟ** | ਅਚਾਨਕ ਲੰਬੀਆਂ ਪ੍ਰਤਿਕ੍ਰਿਆਵਾਂ ਤੋਂ ਬਚਣ ਲਈ `max_tokens` ਸੀਮਾਵਾਂ ਸੈੱਟ ਕਰੋ। |
| **ਬੈਚਿੰਗ** | ਜਿੱਥੇ ਸੰਭਵ ਹੋਵੇ, ਕਈ ਯੂਜ਼ਰ ਕਵੈਰੀਆਂ ਨੂੰ ਇੱਕੋ API ਕਾਲ ਵਿੱਚ ਗਰੁੱਪ ਕਰੋ। |

ਅਮਲੀ ਤੌਰ 'ਤੇ, ਇੱਕ ਤਹਿ-ਕਦਮੀ ਪਹੁੰਚ ਚੰਗੀ ਤਰ੍ਹਾਂ ਕੰਮ ਕਰਦੀ ਹੈ: ਸਧਾਰਨ ਬੇਨਤੀਆਂ ਨੂੰ ਤੇਜ਼, ਸਸਤੇ ਮਾਡਲ ਵੱਲ ਭੇਜੋ ਅਤੇ ਸਿਰਫ਼ ਜਟਿਲ ਕਵੈਰੀਆਂ ਨੂੰ ਹੀ ਹੋਰ ਸਮਰੱਥ ਮਾਡਲ ਤੱਕ ਵਧਾਓ।


## ਸੰਖੇਪ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਇਹ ਸਿੱਖਿਆ ਕਿ:

1. **ਨਿਗਰਾਨੀ ਜੋੜੋ** ਏਜੰਟ ਇੰਟਰੈਕਸ਼ਨਾਂ ਵਿੱਚ ਸਮੇਂ-ਮਾਪ ਅਤੇ ਲੌਗਿੰਗ ਨਾਲ, ਟਰੇਸਿੰਗ ਅਤੇ ਮਾਨੀਟਰਿੰਗ ਲਈ ਬੁਨਿਆਦ ਰੱਖਦੇ ਹੋਏ.
2. **ਏਜੰਟ ਦੇ ਜਵਾਬਾਂ ਦਾ ਮੁਲਾਂਕਣ ਕਰੋ** ਸਵੈਚਾਲਿਤ ਤਰੀਕੇ ਨਾਲ ਇੱਕ ਮੁਲਾਂਕਣ ਕਰਨ ਵਾਲੇ ਏਜੰਟ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਜੋ ਪੂਰਨਤਾ, ਸਹੀਤਾ, ਅਤੇ ਮਦਦਗਾਰਤਾ ਨੂੰ ਸਕੋਰ ਕਰਦਾ ਹੈ.
3. **ਲਾਗਤਾਂ ਨੂੰ ਸੰਭਾਲੋ** ਪ੍ਰਾਂਪਟ ਆਪਟੀਮਾਈਜ਼ੇਸ਼ਨ, ਮਾਡਲ ਚੋਣ, ਕੈਸ਼ਿੰਗ, ਅਤੇ ਟੋਕਨ ਬਜਟਾਂ ਰਾਹੀਂ.

ਇਹ ਉਤਪਾਦਨ ਪੈਟਰਨ ਇਹ ਯਕੀਨੀ ਬਣਾਉਂਦੇ ਹਨ ਕਿ ਤੁਹਾਡੇ AI ਏਜੰਟ ਪੈਮਾਨੇ 'ਤੇ ਭਰੋਸੇਯੋਗ, ਮਾਪਣਯੋਗ, ਅਤੇ ਲਾਗਤ-ਪ੍ਰਭਾਵਸ਼ਾਲੀ ਹਨ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ਅਸਵੀਕਾਰ:
ਇਹ ਦਸਤਾਵੇਜ਼ AI ਅਨੁਵਾਦ ਸੇਵਾ Co‑op Translator (https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦਿਤ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸ਼ੁੱਧਤਾ ਲਈ ਕੋਸ਼ਿਸ਼ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਆਟੋਮੇਟਿਕ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਪਸ਼ਟਤਾਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਮੌਜੂਦ ਦਸਤਾਵੇਜ਼ ਨੂੰ ਹੀ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਮਹੱਤਵਪੂਰਨ ਜਾਂ ਗੰਭੀਰ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਕਾਰਨ ਪੈਦਾਵਾਂ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਅਰਥਾਂ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
